# Normalización

**Capítulo 3 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-modern/batch-norm.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-modern/batch-norm.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Normalización de lotes
<a id="sec_batch_norm"></a>

El entrenamiento de redes neuronales profundas es difícil. Conseguir que converjan en una cantidad razonable de tiempo puede ser complicado. En esta sección, describimos *normalización por lotes (BatchNorm)*, una técnica popular y efectiva que acelera consistentemente la convergencia de redes profundas [Ioffe.Szegedy.2015](https://d2l.ai/chapter_references/zreferences.html). Junto con bloques residuales---cubiertos más tarde en [Referencia sec_resnet](https://d2l.ai/chapter_convolutional-modern/resnet.html#sec-resnet)---la normalización por lotes (BatchNorm) ha hecho posible que los practicantes entren rutinariamente redes con más de 100 capas. Un beneficio secundario (serendípito) de la normalización por lotes (BatchNorm) radica en su regularización inherente.


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## Entrenamiento de redes profundas
Al trabajar con los datos, a menudo preprocesamos antes de la entrenamiento. Las opciones sobre el preprocesamiento de datos a menudo marcan una diferencia enorme en los resultados finales. Recordemos nuestra aplicación de MLPs para predecir los precios de la vivienda ([Referencia sec_kaggle_house](https://d2l.ai/chapter_multilayer-perceptrons/kaggle-house-price.html#sec-kaggle-house)Nuestro primer paso cuando trabajamos con datos reales fue estandarizar nuestras características de entrada para tener cero media $\boldsymbol{\mu} = 0$ y diferencia por unidad $\boldsymbol{\Sigma} = \boldsymbol{1}$ a través de múltiples observaciones [friedman1987exploratory](https://d2l.ai/chapter_references/zreferences.html), frecuentemente reescalando el segundo para que la diagonal sea unidad, es decir, $\Sigma_{ii} = 1$. Otra estrategia es volver a escalar los vectores a la longitud de la unidad, posiblemente cero media *por observación*. Esto puede funcionar bien, por ejemplo, para los datos del sensor espacial. Estas técnicas de preprocesamiento y muchas otras, son beneficiosas para mantener el problema de estimación bien controlado. [guyon2008feature](https://d2l.ai/chapter_references/zreferences.html), por ejemplo. La estandarización de vectores también tiene el efecto secundario agradable de limitar la complejidad de la función de las funciones que actúan sobre él. Por ejemplo, el famoso radio-margen encuadernado [Vapnik95](https://d2l.ai/chapter_references/zreferences.html) en máquinas vectoriales de soporte y el Teorema de Convergencia de Perceptron [Novikoff62](https://d2l.ai/chapter_references/zreferences.html) se basan en aportaciones de norma limitada.

Intuitivamente, esta estandarización juega bien con nuestros optimizadores ya que pone los parámetros *a priori* en una escala similar. Como tal, es natural preguntar si un paso de normalización correspondiente *dentro* una red profunda podría no ser beneficiosa. Si bien este no es el razonamiento que llevó a la invención de la normalización por lotes (BatchNorm) [Ioffe.Szegedy.2015](https://d2l.ai/chapter_references/zreferences.html), es una forma útil de entenderla y su primo, normalización de capas [Ba.Kiros.Hinton.2016](https://d2l.ai/chapter_references/zreferences.html), dentro de un marco unificado.

En segundo lugar, para un MLP típico o CNN, a medida que entrenamos, las variables en capas intermedias (por ejemplo, salidas de transformación afín en MLP) pueden tomar valores con magnitudes muy diversas: si a lo largo de las capas de entrada a salida, a través de unidades en la misma capa, y con el tiempo debido a nuestras actualizaciones a los parámetros del modelo. Los inventores de normalización por lotes (BatchNorm) postularon informalmente que esta deriva en la distribución de tales variables podría obstaculizar la convergencia de la red. Intuitivamente, podríamos conjeturar que si una capa tiene activaciones variables que son 100 veces la de otra capa, esto podría requerir ajustes compensatorios en las tasas de aprendizaje. Resolvedores adaptativos como AdaGrad [Duchi.Hazan.Singer.2011](https://d2l.ai/chapter_references/zreferences.html), Adam [Kingma.Ba.2014](https://d2l.ai/chapter_references/zreferences.html), Yogi [Zaheer.Reddi.Sachan.ea.2018](https://d2l.ai/chapter_references/zreferences.html), o Shampoo [anil2020scalable](https://d2l.ai/chapter_references/zreferences.html) Distributed tienen como objetivo abordar esto desde el punto de vista de la optimización, e.g., añadiendo aspectos de métodos de segundo orden. La alternativa es evitar que el problema ocurra, simplemente por normalización adaptativa.

En tercer lugar, las redes más profundas son complejas y tienden a ser más susceptibles de sobreadaptarse, lo que significa que la regularización se vuelve más crítica. Una técnica común para regularización es la inyección de ruido. Esto se conoce desde hace mucho tiempo, por ejemplo, con respecto a la inyección de ruido para las entradas [Bishop.1995](https://d2l.ai/chapter_references/zreferences.html). También forma la base de la dropout en [Referencia sec_dropout](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html#sec-dropout). Como resulta, muy serendipitalmente, la normalización por lotes (BatchNorm) transmite los tres beneficios: preprocesamiento, estabilidad numérica y regularización.

La normalización por lotes (BatchNorm) se aplica a las capas individuales, o opcionalmente, a todas ellas: En cada iteración de entrenamiento, primero normalizamos las entradas (de normalización por lotes (BatchNorm)) restando su media y dividiendo por su desviación estándar, donde ambas se estiman sobre la base de las estadísticas del minibatch actual. A continuación, aplicamos un coeficiente de escala y un desplazamiento para recuperar los grados perdidos de libertad. Es precisamente debido a esta *normalización* basada en estadísticas *batch* que *normalización por lotes (BatchNorm)* deriva su nombre.

Tenga en cuenta que si tratamos de aplicar la normalización por lotes (BatchNorm) con minibatches de tamaño 1, no seríamos capaces de aprender nada. Esto se debe a que después de restar los medios, cada unidad oculta tomaría valor 0. Como usted podría adivinar, ya que estamos dedicando una sección entera a la normalización por lotes (BatchNorm), con minibatches suficientemente grandes el enfoque resulta eficaz y estable. Una ventaja aquí es que al aplicar la normalización por lotes (BatchNorm), la elección del tamaño del lote es aún más importante que sin normalización por lotes (BatchNorm), o al menos, se necesita una calibración adecuada, ya que podríamos ajustar el tamaño del lote.

Denotar por $\mathcal{B}$ un minibatch y dejar que $\mathbf{x} \in \mathcal{B}$ sea una entrada a la normalización por lotes (BatchNorm) ($\textrm{BN}$). En este caso la normalización por lotes (BatchNorm) se define como sigue:

$$\textrm{BN}(\mathbf{x}) = \boldsymbol{\gamma} \odot \frac{\mathbf{x} - \hat{\boldsymbol{\mu}}_\mathcal{B}}{\hat{\boldsymbol{\sigma}}_\mathcal{B}} + \boldsymbol{\beta}.$$

:eqlabel:`eq_batchnorm`

En [Referencia eq_batchnorm](https://d2l.ai/#eq-batchnorm), $\hat{\boldsymbol{\mu}}_\mathcal{B}$ es la media de la muestra y $\hat{\boldsymbol{\sigma}}_\mathcal{B}$ es la desviación estándar de la muestra del minibatch $\mathcal{B}$. Después de aplicar estandarización, el minibatch resultante tiene media cero y varianza de unidad. La elección de la varianza de unidad (en lugar de algún otro número mágico) es arbitraria. Recuperamos este grado de libertad mediante la inclusión de un parámetro *escala de elementos* $\boldsymbol{\gamma}$ y un parámetro *desplazamiento* $\boldsymbol{\beta}$ que tienen la misma forma que $\mathbf{x}$. Ambos son parámetros que necesitan ser aprendidos como parte del entrenamiento del modelo.

Las magnitudes variables de las capas intermedias no pueden diferir durante el entrenamiento, ya que la normalización por lotes (BatchNorm) se centra activamente y las redimensiona de nuevo a una media y tamaño dados (vía $\hat{\boldsymbol{\mu}}_\mathcal{B}$ y ${\hat{\boldsymbol{\sigma}}_\mathcal{B}}$). La experiencia práctica confirma que, como se aludió a la discusión de la modificación de características, la normalización por lotes (BatchNorm) parece permitir tasas de aprendizaje más agresivas. Calculamos $\hat{\boldsymbol{\mu}}_\mathcal{B}$ y ${\hat{\boldsymbol{\sigma}}_\mathcal{B}}$ en [Referencia eq_batchnorm](https://d2l.ai/#eq-batchnorm) de la siguiente manera:

$$\hat{\boldsymbol{\mu}}_\mathcal{B} = \frac{1}{|\mathcal{B}|} \sum_{\mathbf{x} \in \mathcal{B}} \mathbf{x}
\textrm{ and }
\hat{\boldsymbol{\sigma}}_\mathcal{B}^2 = \frac{1}{|\mathcal{B}|} \sum_{\mathbf{x} \in \mathcal{B}} (\mathbf{x} - \hat{\boldsymbol{\mu}}_{\mathcal{B}})^2 + \epsilon.$$

Tenga en cuenta que añadimos una pequeña constante $\epsilon > 0$ a la estimación de la varianza para asegurar que nunca intentemos la división por cero, incluso en los casos en que la estimación de la varianza empírica podría ser muy pequeña o desaparecer. Las estimaciones $\hat{\boldsymbol{\mu}}_\mathcal{B}$ y ${\hat{\boldsymbol{\sigma}}_\mathcal{B}}$ contrarrestan el problema de la escala mediante el uso de estimaciones ruidosas de la media y la varianza.

Esto resulta ser un tema recurrente en el aprendizaje profundo. Por razones que aún no están bien caracterizadas teóricamente, diversas fuentes de ruido en la optimización a menudo conducen a un entrenamiento más rápido y menos sobrefitting: esta variación parece actuar como una forma de regularización.
[Teye.Azizpour.Smith.2018](https://d2l.ai/chapter_references/zreferences.html) y [Luo.Wang.Shao.ea.2018](https://d2l.ai/chapter_references/zreferences.html)
Este tamaño particular de minibatch parece inyectar sólo la "cantidad correcta" de ruido por capa, tanto en términos de escala a través de $\hat{\boldsymbol{\sigma}}$, como en términos de compensación a través de $\hat{\boldsymbol{\mu}}$: un minibatch más grande regulariza menos debido a las estimaciones más estables, mientras que pequeños minibatch destruyen la señal útil debido a la alta varianza. Explorando esta dirección más adelante, considerando los tipos alternativos de preprocesamiento y filtrado puede conducir a otros tipos eficaces de regularización.

Fijando un modelo entrenado, usted podría pensar que preferiríamos usar todo el conjunto de datos para estimar la media y la varianza. Una vez que el entrenamiento esté completo, ¿por qué querríamos que la misma imagen se clasificase de manera diferente, dependiendo del lote en el que resida? Durante el entrenamiento, tal cálculo exacto es inviable porque las variables intermedias para todos los ejemplos de datos cambian cada vez que actualizamos nuestro modelo. Sin embargo, una vez que el modelo es entrenado, podemos calcular los medios y varianzas de las variables de cada capa basándose en todo el conjunto de datos. De hecho, esta es la práctica estándar para modelos que emplean la normalización por lotes (BatchNorm); así las capas de normalización por lotes (BatchNorm) funcionan de manera diferente en *modo de entrenamiento* (normalización mediante estadísticas de minibatch) que en *modo de predicción* (normalización mediante estadísticas de conjuntos de datos). [Referencia sec_dropout](https://d2l.ai/chapter_multilayer-perceptrons/dropout.html#sec-dropout), donde el ruido sólo se inyecta durante el entrenamiento.

## Capas de normalización por lotes (BatchNorm)
Las implementaciones de normalización por lotes (BatchNorm) para capas totalmente conectadas y capas convolucionales son ligeramente diferentes. Una diferencia clave entre la normalización por lotes (BatchNorm) y otras capas es que debido a que la primera opera en un minibatch completo a la vez, no podemos ignorar la dimensión de lotes como lo hicimos antes al introducir otras capas.

### Capas totalmente conectadas
Al aplicar la normalización por lotes (BatchNorm) a capas totalmente conectadas,
[Ioffe.Szegedy.2015](https://d2l.ai/chapter_references/zreferences.html), en su artículo original, situaron la normalización por lotes (BatchNorm) después de la transformación afín
y *antes* de la función de activación no lineal. Más tarde las aplicaciones experimentaron con la inserción de normalización por lotes (BatchNorm) a la derecha *después* funciones de activación. Denotando la entrada a la capa totalmente conectada por $\mathbf{x}$, la transformación afín por $\mathbf{W}\mathbf{x} + \mathbf{b}$ (con el parámetro de peso $\mathbf{W}$ y el parámetro de sesgo $\mathbf{b}$), y la función de activación por $\phi$, podemos expresar el cálculo de una salida de la capa $\mathbf{h}$ de lote-normalización-activada, totalmente conectada de la siguiente manera:

$$\mathbf{h} = \phi(\textrm{BN}(\mathbf{W}\mathbf{x} + \mathbf{b}) ).$$

Recordemos que la media y la varianza se calculan en el minibatch *same* en el que se aplica la transformación.

### Capas convolucionales
Del mismo modo, con las capas convolucionales, podemos aplicar la normalización por lotes (BatchNorm) después de la convolución pero antes de la función de activación no lineal. La diferencia clave de la normalización por lotes (BatchNorm) en capas totalmente conectadas es que aplicamos la operación sobre una base por canal * a través de todas las ubicaciones *. Esto es compatible con nuestra suposición de invarianza de traducción que llevó a las convoluciones: asumimos que la ubicación específica de un patrón dentro de una imagen no era crítica para el propósito de entender.

Supongamos que nuestros minibatches contienen ejemplos $m$ y que para cada canal, la salida de la convolución tiene altura $p$ y anchura $q$. Para las capas convolucionales, realizamos la normalización de cada lote sobre los elementos $m \cdot p \cdot q$ por canal de salida simultáneamente. Así, recopilamos los valores sobre todas las ubicaciones espaciales al calcular la media y la varianza y consecuentemente aplicamos la misma media y varianza dentro de un canal dado para normalizar el valor en cada ubicación espacial. Cada canal tiene sus propios parámetros de escala y cambio, ambos son escalares.

### Normalización de la capa
<a id="subsec_layer-normalization-in-bn"></a>

Tenga en cuenta que en el contexto de las convoluciones la normalización por lotes (BatchNorm) está bien definida incluso para minibatches de tamaño 1: después de todo, tenemos todas las ubicaciones a través de una imagen a promedio. Por lo tanto, la media y la varianza están bien definidas, incluso si está dentro de una sola observación. Esta consideración llevó a [Ba.Kiros.Hinton.2016](https://d2l.ai/chapter_references/zreferences.html) a introducir la noción de *normalización de capa*. Funciona como una norma de lote, sólo que se aplica a una observación a la vez. En consecuencia, tanto el desplazamiento como el factor de escala son escalares. Para un vector $n$-dimensional $\mathbf{x}$, las normas de capa son dadas por

$$\mathbf{x} \rightarrow \textrm{LN}(\mathbf{x}) =  \frac{\mathbf{x} - \hat{\mu}}{\hat\sigma},$$

donde la escala y la compensación se aplican en función del coeficiente y dado por

$$\hat{\mu} \stackrel{\textrm{def}}{=} \frac{1}{n} \sum_{i=1}^n x_i \textrm{ and }
\hat{\sigma}^2 \stackrel{\textrm{def}}{=} \frac{1}{n} \sum_{i=1}^n (x_i - \hat{\mu})^2 + \epsilon.$$

Como antes añadimos un pequeño offset $\epsilon > 0$ para evitar la división por cero. Uno de los principales beneficios de usar la normalización de capas es que evita divergencias. Después de todo, ignorando $\epsilon$, la salida de la normalización de capas es independiente de escala. Es decir, tenemos $\textrm{LN}(\mathbf{x}) \approx \textrm{LN}(\alpha \mathbf{x})$ para cualquier elección de $\alpha \neq 0$. Esto se convierte en una igualdad para $|\alpha| \to \infty$ (la igualdad aproximada se debe al offset $\epsilon$ para la varianza).

Otra ventaja de la normalización de la capa es que no depende del tamaño del minibatch. También es independiente de si estamos en régimen de entrenamiento o de prueba. En otras palabras, es simplemente una transformación determinista que estandariza las activaciones a una escala dada. Esto puede ser muy beneficioso para prevenir divergencias en la optimización. Omitimos más detalles y recomendamos que los lectores interesados consulten el artículo original.

### Normalización de lotes durante la predicción
Como mencionamos anteriormente, la normalización por lotes (BatchNorm) normalmente se comporta de manera diferente en el modo de entrenamiento que en el modo de predicción. Primero, el ruido en la media de la muestra y la varianza de la muestra que surge de estimar cada uno en minibatches ya no es deseable una vez que hemos entrenado el modelo. Segundo, podríamos no tener el lujo de calcular estadísticas de normalización por lotes (BatchNorm). Por ejemplo, podríamos necesitar aplicar nuestro modelo para hacer una predicción a la vez.

Típicamente, después del entrenamiento, utilizamos todo el conjunto de datos para calcular estimaciones estables de las estadísticas variables y luego fijarlas en el tiempo de predicción. Por lo tanto, la normalización por lotes (BatchNorm) se comporta de manera diferente durante el entrenamiento que durante la evaluación. Recordemos que la dropout también exhibe esta característica.

## Implementación desde cero

Para ver cómo funciona la normalización por lotes (BatchNorm) en la práctica, implementamos uno desde cero abajo.


In [ ]:
def batch_norm(X, gamma, beta, moving_mean, moving_var, eps, momentum):
    # Usar is_grad_habilitado para determinar si estamos en modo de entrenamiento
    if not torch.is_grad_enabled():
        # En el modo de predicción, utilizar la media y la varianza obtenidas mediante el promedio móvil
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
    else:
        assert len(X.shape) in (2, 4)
        if len(X.shape) == 2:
            # Cuando utilice una capa totalmente conectada, calcule la media y
            # varianza en la dimensión de las características
            mean = X.mean(dim=0)
            var = ((X - mean) ** 2).mean(dim=0)
        else:
            # Al usar una capa convolucional bidimensional, calcule el
            # media y varianza en la dimensión del canal (eje=1).
            # necesidad de mantener la forma de X, para que la transmisión
            # operación se puede llevar a cabo más tarde
            mean = X.mean(dim=(0, 2, 3), keepdim=True)
            var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)
        # En el modo de entrenamiento, se utilizan la media actual y la varianza
        X_hat = (X - mean) / torch.sqrt(var + eps)
        # Actualizar la media y la varianza usando el promedio móvil
        moving_mean = (1.0 - momentum) * moving_mean + momentum * mean
        moving_var = (1.0 - momentum) * moving_var + momentum * var
    Y = gamma * X_hat + beta  # Escala y desplazamiento
    return Y, moving_mean.data, moving_var.data

Ahora podemos **crear una capa `BatchNorm` adecuada.**Nuestra capa mantendrá parámetros adecuados para la escala `gamma` y el cambio `beta`, los cuales se actualizarán durante el entrenamiento. Además, nuestra capa mantendrá promedios móviles de los medios y variaciones para su uso posterior durante la predicción del modelo.

Dejando a un lado los detalles algorítmicos, note el patrón de diseño subyacente a nuestra implementación de la capa. Típicamente, definimos las matemáticas en una función separada, digamos `batch_norm`. Luego integramos esta funcionalidad en una capa personalizada, cuyo código se ocupa principalmente de asuntos de contabilidad, como mover datos al contexto del dispositivo correcto, asignar e inicializar cualquier variable requerida, mantener un registro de promedios móviles (aquí para la media y la varianza), etc. Este patrón permite una separación limpia de matemáticas del código de la placa de caldera. También tenga en cuenta que por conveniencia no nos preocupamos de inferir automáticamente la forma de entrada aquí; por lo tanto necesitamos especificar el número de características a lo largo de todo. Por ahora todas las bibliotecas de aprendizaje profundo modernos ofrecen detección automática de tamaño y forma en las APIs de normalización por lotes (BatchNorm) de alto nivel (en la práctica usaremos esto en su lugar).


In [ ]:
class BatchNorm(nn.Module):
    # num_features: el número de salidas para una capa totalmente conectada o la
    # número de canales de salida para una capa convolucional. num_dims: 2 para una
    # capa totalmente conectada y 4 para una capa convolucional
    def __init__(self, num_features, num_dims):
        super().__init__()
        if num_dims == 2:
            shape = (1, num_features)
        else:
            shape = (1, num_features, 1, 1)
        # El parámetro escala y el parámetro cambio (parámetros modelo) son
        # inicializado a 1 y 0, respectivamente
        self.gamma = nn.Parameter(torch.ones(shape))
        self.beta = nn.Parameter(torch.zeros(shape))
        # Las variables que no son parámetros de modelo se inicializan a 0 y
        # 1
        self.moving_mean = torch.zeros(shape)
        self.moving_var = torch.ones(shape)

    def forward(self, X):
        # Si X no está en la memoria principal, copiar move_mean y move_var a
        # el dispositivo donde se encuentra X
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)
        # Guardar la imagen actualizada move_mean y move_var
        Y, self.moving_mean, self.moving_var = batch_norm(
            X, self.gamma, self.beta, self.moving_mean,
            self.moving_var, eps=1e-5, momentum=0.1)
        return Y

Utilizamos `momentum` para gobernar la agregación sobre la media pasada y las estimaciones de varianza. Esto es algo de un mal nombre, ya que no tiene nada que ver con el término *momentum* de optimización. Sin embargo, es el nombre comúnmente adoptado para este término y en deferencia a la convención de nombres API usamos el mismo nombre de variable en nuestro código.

## LeNet con normalización por lotes (BatchNorm)

Para ver cómo aplicar `BatchNorm` en contexto, a continuación lo aplicamos a un modelo tradicional de LeNet ([Referencia sec_lenet](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html#sec-lenet)). Recuerde que la normalización por lotes (BatchNorm) se aplica después de las capas convolucionales o capas totalmente conectadas pero antes de las funciones de activación correspondientes.


In [ ]:
class BNLeNetScratch(d2l.Classifier):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.LazyConv2d(6, kernel_size=5), BatchNorm(6, num_dims=4),
            nn.Sigmoid(), nn.AvgPool2d(kernel_size=2, stride=2),
            nn.LazyConv2d(16, kernel_size=5), BatchNorm(16, num_dims=4),
            nn.Sigmoid(), nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Flatten(), nn.LazyLinear(120),
            BatchNorm(120, num_dims=2), nn.Sigmoid(), nn.LazyLinear(84),
            BatchNorm(84, num_dims=2), nn.Sigmoid(),
            nn.LazyLinear(num_classes))

Como antes, **entrenaremos nuestra red en el conjunto de datos de Fashion-MNIST**. Este código es virtualmente idéntico a cuando entrenamos por primera vez a LeNet.


### Nota docente de Hespérides

BatchNorm estima estadísticas por característica a partir del lote y cambia de comportamiento entre entrenamiento y evaluación. LayerNorm normaliza las características de cada ejemplo o posición y no necesita estadísticas corrientes del lote. Por eso no basta con sustituir nombres: identifica los ejes normalizados. En una evaluación reproducible, combina `model.eval()` con `torch.no_grad()`; lo primero cambia capas como dropout y BatchNorm, lo segundo desactiva la construcción del grafo.

Vínculo con los apuntes: sesión 3, «Normalización».


In [ ]:
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128)
model = BNLeNetScratch(lr=0.1)
model.apply_init([next(iter(data.get_dataloader(True)))[0]], d2l.init_cnn)
trainer.fit(model, data)

Echemos un vistazo al parámetro escala `gamma` y al parámetro cambio `beta`** aprendido de la primera capa de normalización por lotes (BatchNorm).


In [ ]:
model.net[1].gamma.reshape((-1,)), model.net[1].beta.reshape((-1,))

## Implementación concisa

Comparado con la clase `BatchNorm`, que acabamos de definir, podemos utilizar la clase `BatchNorm` definida en API de alto nivel desde el biblioteca de aprendizaje profundo directamente. El código se ve virtualmente idéntico a nuestra implementación anterior, excepto que ya no necesitamos proporcionar argumentos adicionales para obtener las dimensiones correctas.


In [ ]:
class BNLeNet(d2l.Classifier):
    def __init__(self, lr=0.1, num_classes=10):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.LazyConv2d(6, kernel_size=5), nn.LazyBatchNorm2d(),
            nn.Sigmoid(), nn.AvgPool2d(kernel_size=2, stride=2),
            nn.LazyConv2d(16, kernel_size=5), nn.LazyBatchNorm2d(),
            nn.Sigmoid(), nn.AvgPool2d(kernel_size=2, stride=2),
            nn.Flatten(), nn.LazyLinear(120), nn.LazyBatchNorm1d(),
            nn.Sigmoid(), nn.LazyLinear(84), nn.LazyBatchNorm1d(),
            nn.Sigmoid(), nn.LazyLinear(num_classes))

A continuación, ** utilizamos los mismos hiperparametros para entrenar nuestro modelo.** Tenga en cuenta que, como de costumbre, la variante API de alto nivel se ejecuta mucho más rápido porque su código ha sido compilado a C++ o CUDA, mientras que nuestra implementación personalizada debe ser interpretada por Python.


In [ ]:
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
data = d2l.FashionMNIST(batch_size=128)
model = BNLeNet(lr=0.1)
model.apply_init([next(iter(data.get_dataloader(True)))[0]], d2l.init_cnn)
trainer.fit(model, data)

## Discusión
Intuitivamente, la normalización por lotes (BatchNorm) se piensa para hacer el paisaje de optimización más suave. Sin embargo, debemos tener cuidado de distinguir entre intuiciones especulativas y verdaderas explicaciones para los fenómenos que observamos cuando entrenamos modelos profundos. Recordemos que ni siquiera sabemos por qué las redes neuronales profundas más simples (PLM y CNNs convencionales) se generalizan bien en primer lugar. Incluso con la dropout y la decaimiento de pesos, siguen siendo tan flexibles que su capacidad de generalizar a datos invisibles probablemente necesita garantías de generalización de aprendizaje-teorética significativamente más refinadas.

El documento original que proponía la normalización por lotes (BatchNorm) [Ioffe.Szegedy.2015](https://d2l.ai/chapter_references/zreferences.html), además de introducir una herramienta poderosa y útil, ofreció una explicación de por qué funciona: reduciendo *cambio covariable interno*. Presumiblemente por *cambio covariable interno* significaron algo como la intuición expresada arriba---la noción de que la distribución de valores variables cambia en el curso de la formación. Sin embargo, hubo dos problemas con esta explicación: i) Esta deriva es muy diferente de *cambio covariable*, haciendo que el nombre sea un mal nombre. Si algo, está más cerca de la deriva conceptual. ii) La explicación ofrece una intuición sub-especificada pero deja la pregunta de *por qué precisamente esta técnica funciona* una pregunta abierta que quiere una explicación rigurosa. A lo largo de este libro, pretendemos transmitir las intuiciones que los profesionales utilizan para guiar su desarrollo de redes neuronales profundas. Sin embargo, creemos que es importante separar estas intuiciones orientadoras de hechos científicos establecidos.

Tras el éxito de la normalización por lotes (BatchNorm), su explicación en términos de *cambio covariable interno* ha surgido repetidamente en debates en la literatura técnica y discurso más amplio sobre cómo presentar la investigación de aprendizaje automático. En un discurso memorable dado al aceptar un Premio Test of Time en la conferencia NeurIPS 2017, Ali Rahimi utilizó *cambio covariable interno* como un punto focal en un argumento que compara la práctica moderna de aprendizaje profundo con la alquimia. Posteriormente, el ejemplo fue revisado en detalle en un documento de posición que esboza tendencias preocupantes en el aprendizaje automático [Lipton.Steinhardt.2018](https://d2l.ai/chapter_references/zreferences.html). Otros autores han propuesto explicaciones alternativas para el éxito de la normalización por lotes (BatchNorm), algunos [Santurkar.Tsipras.Ilyas.ea.2018](https://d2l.ai/chapter_references/zreferences.html) alegando que el éxito de la normalización por lotes (BatchNorm) viene a pesar de mostrar un comportamiento que es en cierto modo opuesto al que se afirma en el documento original.

Observamos que el *cambio covariable interno* no es más digno de crítica que cualquiera de las miles de afirmaciones igualmente vagas que se hacen cada año en la literatura técnica de aprendizaje automático. Probablemente, su resonancia como punto focal de estos debates se debe a su amplia capacidad de reconocimiento para el público objetivo. La normalización por lotes (BatchNorm) ha demostrado ser un método indispensable, aplicado en casi todos los clasificadores de imágenes desplegados, obteniendo el papel que introdujo la técnica decenas de miles de citas. Sin embargo, conjeturamos que los principios rectores de la regularización a través de la inyección de ruido, la aceleración a través de la recalificación y, por último, el preprocesamiento pueden conducir a nuevas invenciones de capas y técnicas en el futuro.

En una nota más práctica, hay una serie de aspectos que vale la pena recordar sobre la normalización por lotes (BatchNorm):

* Durante el entrenamiento del modelo, la normalización por lotes (BatchNorm) ajusta continuamente la salida intermedia de
la red utilizando la media y la desviación estándar del minibatch, de manera que los valores de la salida intermedia en cada capa a lo largo de la red neural sean más estables.
* La normalización por lotes (BatchNorm) es ligeramente diferente para las capas totalmente conectadas que para las capas convolucionales.
para las capas convolucionales, la normalización de capas a veces se puede utilizar como una alternativa.
* Como una capa de dropout, las capas de normalización por lotes (BatchNorm) tienen diferentes comportamientos
en modo de entrenamiento que en modo de predicción.
* La normalización por lotes (BatchNorm) es útil para regularizar y mejorar la convergencia en la optimización.
la motivación original de reducir el cambio covariable interno no parece ser una explicación válida.
* Para modelos más robustos que sean menos sensibles a las perturbaciones de entrada, considere eliminar la normalización por lotes (BatchNorm) [wang2022removing](https://d2l.ai/chapter_references/zreferences.html).

## Ejercicios
1. ¿Debemos eliminar el parámetro sesgo de la capa totalmente conectada o la capa convolucional antes de la normalización por lotes (BatchNorm)? ¿Por qué?
1. Compare las tasas de aprendizaje de LeNet con y sin normalización por lotes (BatchNorm).
    1. Trazar el aumento en la precisión de validación.
    1. ¿Qué tan grande puede ser la tasa de aprendizaje antes de que la optimización falle en ambos casos?
1. ¿Necesitamos normalización por lotes (BatchNorm) en cada capa?
1. Implementar una versión "lite" de normalización por lotes (BatchNorm) que sólo elimina la media, o alternativamente una que sólo elimina la varianza. ¿Cómo se comporta?
1. Fijar los parámetros `beta` y `gamma`. Observar y analizar los resultados.
1. ¿Puede reemplazar el dropout por normalización por lotes (BatchNorm)? ¿Cómo cambia el comportamiento?
1. Ideas de investigación: piensa en otras transformaciones de normalización que puedes aplicar:
    1. ¿Se puede aplicar la transformación integral de probabilidad?
    1. ¿Puedes usar una estimación de covarianza de rango completo? ¿Por qué probablemente no harías eso?
    1. ¿Puede utilizar otras variantes de matriz compacta (bloque-diagonal, rango de baja desplazamiento, Monarca, etc.)?
    1. ¿Una compresión de la esparificación actúa como un regularizador?
    1. ¿Hay otras proyecciones (por ejemplo, cono convexo, transformaciones de grupos específicos de simetría) que pueda utilizar?


[Debate del original](https://discuss.d2l.ai/t/84)
